<span style="font-size:10pt">DIADEM course "Deep Learning - Image Classification & objects detection" -- Mai 20-21, 2026 - v1.1<br> 
CC BY-SA 4.0 Jean-Luc CHARLES (Jean-Luc.charles@mailo.com)</span>

<span style="color:Sienna; font-family:arial;font-size:1.2cm; font-weight:bold">
    The question of Reproducibility<br>
    in Supervised Learning
</span>

<span style="color:Sienna; font-family:arial; font-size:15pt;">
    In this notebook we use a simple Dense Neural Network (DNN) model<br>
    to focus on the reproducibility of the training process.
</span>

# Preliminaries

## Import the Python modules

In [ ]:
import os
# suppress tensorflow verbose warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# Deep Learning modules:
import tensorflow as tf
from tensorflow import keras

# General modules:
import numpy as np
import matplotlib.pyplot as plt
from time import time
from pathlib import Path
from cpuinfo import get_cpu_info
import GPUtil
import sys

# local modules:
from utils.tools import elapsed_time_since, cpu_gpu, plot_loss_accuracy

In [ ]:
print(f"Python    : {sys.version.split()[0]}")
print(f"tensorflow: {tf.__version__} with keras {keras.__version__}")
print(f"numpy     : {np.__version__}")

## Global settings:

In [ ]:
# To visualize the graphs directly in the cell of the Notebook
%matplotlib inline

# SEED will be used to set the "seed" of the random generators in order
# to have repeatable random numbers
SEED = 1234

# Ask tensorflow to display only ERROR messages;
tf.get_logger().setLevel('ERROR')

## Check wether GPU is available for tensorflow or not:

In [ ]:
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"tensorflow running on {len(gpus)} GPU(s):")
    for gpu in gpus:
        print(f"  - {gpu.name}")
        # configure tensorflow to dynamically allocate GPU memory as needed:
        tf.config.experimental.set_memory_growth(gpu, True)
else:
    print("tensorflow running on CPU.")

## Create the `model` directory

In [ ]:
print(f'{"WORKING DIRECTORY":17s}: {Path.cwd()}')

model_dir = Path("./model")
model_dir.mkdir(exist_ok=True)
print(f'{"MODEL DIRECTORY":17s}: {model_dir.absolute()}')

# 1 - Prepare the MNIST dataset

## 1.1 Load the MNIST dataset (images and labels)

We use the keras `load_data` function to load the data from the MNIST 
(see [tf.keras.datasets.mnist.load_data](https://www.tensorflow.org/api_docs/python/tf/keras/datasets/mnist/load_data)):<br>
- `train_img`, `train_lab` are the training images and labels,
- `valid_img`, `valid_lab` are the validation images and labels.

In [ ]:
(train_img, train_lab), (valid_img, valid_lab) = keras.datasets.mnist.load_data()

Let's check the `shape` and `dtype` attributes of the numpy arrays:

In [ ]:
print(f"train_img.shape: {train_img.shape}, dtype: {train_img.dtype}")
print(f"train_lab.shape: {train_lab.shape}, dtype: {train_lab.dtype}")
print(f"valid_img.shape: {valid_img.shape}, dtype: {valid_img.dtype}")
print(f"lab_vaild.shape: {valid_lab.shape}, dtype: {valid_lab.dtype}")

<span style="color:Sienna; font-family:arial; font-size:12pt;">
    To avoid long computation time, we take only 1000 images & labels in the training dataset<br>
    and 200 images & labels in the validation dataset.
</span>

In [ ]:
train_img = train_img[:1000]
valid_img = valid_img[:200]

train_lab = train_lab[:1000]
valid_lab = valid_lab[:200]

print(f'{train_img.shape=}')
print(f'{valid_img.shape=}')

## 1.2 Display images and labels:

#### Simple display of a MNIST image with the plt.plot function

In [ ]:
plt.figure(figsize=(1,1))
plt.imshow(train_img[199], cmap='gray')
plt.axis('off');

#### Display images on a grid with labels

The local module `utils.tools` defines the `plot_images` function which can be used to display a grid of MNIST images:

In [ ]:
from utils.tools import display_image
help(display_image)

<span style="color:Blue; font-family:arial; font-size:11pt;">
    With the <span style="font-family:monospace; background-color: #EBEBEB;">&thinsp;display_image&thinsp;</span> function, print a grid of 4 x 10 images, starting at the rank you want, with the display of the image labels:
</span>

## 1.5 -  Define global parameters

To avoid hard-coding global parameters such as the number of train/valid/test images, or the size of the images, these parameters are retrieved from the data set:
- with the shape attribute of the `train_img` and `test_im` arrays
- with the size attribute of the first training image for example...


<span style="color:Blue; font-family:arial; font-size:11pt;">
    Complete the cell below to define useful global parameters:
</span>

In [ ]:
# check:
print(f"{NB_TRAIN_IMG} training images, {NB_VALID_IMG} validation images")
print(f"{train_img.shape[1]}x{train_img.shape[2]}={NB_PIXEL} pixels in each image")
print(f"{NB_CLASS} classes found in the `train_lab` ndarray")

# 2 - Process input data

Two treatments must be applied to the data from the MNIST database:
- **on images**: convert the matrices of  28$\,\times\,$28 pixels (`uint8`integers) into **normalized** vectors $(V_i)_{i=0..783}$ of 784 real values $V_i$ with $ 0 \leqslant V_i \leqslant 1$;
- **on labels**: transform integers into *one-hot* vectors.

## 2.1 - Transform input matrices into normalized vectors

We define the arrays `x_train`, `x_valid` containing the matrices of the arrays `train_img`, `valid_img` *flattened* to normalized vectors (values between 0 and 1):

In [ ]:
x_train = train_img.reshape(NB_TRAIN_IMG, NB_PIXEL)/255
x_valid = valid_img.reshape(NB_VALID_IMG, NB_PIXEL)/255

<span style="color:Blue; font-family:arial; font-size:11pt;">
    Check the shape, min and max of <span style="font-family:monospace; background-color: #EBEBEB;">&thinsp;x_train&thinsp;</span> and <span style="font-family:monospace; background-color: #EBEBEB;">&thinsp;x_valid&thinsp;</span>:
</span>

## 2.2 - *one-hot* encoding of labels

We use the **tf.keras** `to_categorical` function (see [tf.keras.utils.to_categorical](https://www.tensorflow.org/api_docs/python/tf/keras/utils/to_categorical)) to define the `y_train` and `y_valid` arrays containing the *hot-one* encoded version of `train_lab` and `valid_lab`:

In [ ]:
from tensorflow.keras.utils import to_categorical

# 'one-hot' encoding' of labels :
y_train = to_categorical(train_lab)
y_valid = to_categorical(valid_lab)

<span style="color:Blue; font-family:arial; font-size:11pt;">
    Print the first 10 values of <span style="font-family:monospace; background-color: #EBEBEB;">&thinsp;train_lab&thinsp;</span> and <span style="font-family:monospace; background-color: #EBEBEB;">&thinsp;y_train&thinsp;</span>:
</span>

# 3 - Build the Dense Neural Network (DNN)

 Now we build a simple  **Dense Neural Network** to classify the MNIST images.<br>
 Of course, this is not the "state of the art" : convolutive NN, transformers have much more impressive scores to classify images,<br>
 but we just want want a simple model easy to understand with short training computation time.<br><br>
We build this simple DNN:
- an **input layer** of 784 values (the pixel matrices of the 28 $\times$ 28 MNIST images flattened into normalized vectors of 784  `float` numbers),
- a **hidden layer** of 784 neurons using the `relu` activation function,
- an **output layer** of 10 neurons, for the classification of the 10 digits {0,1,2...9}, using the `softmax` activation function required for classifying.

<p style="text-align:center; font-style:italic; font-size:12px;">
      <img src="img/simple-DNN.png" alt="simple-DNN.png" style="width:900px;"><br>
     [image: JLC]
</p>

For the sake of convenience we défine a parameterized function to build the DNN: 

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, Dropout

def build_DNN(nb_input, nb_neuron, nb_class, seed=None, name=''):

    if seed is not None:
        ##########################
        # Deterministic training #
        ##########################
        # 1/ set the seed of the random generators used by tensorflow:
        tf.keras.utils.set_random_seed(seed)
        # 2/ make the tf ops determinisctic 
        # [see https://blog.tensorflow.org/2022/05/whats-new-in-tensorflow-29.html]
        tf.config.experimental.enable_op_determinism() 

    model = Sequential()
    model.add(Input(shape=(nb_input,), name='input'))               # INPUT layer
    model.add(Dense(nb_neuron, activation='relu', name='h1'))       # HIDDEN layer
    model.add(Dense(nb_class, activation='softmax', name='output')) # OUTPUT layer
   
    model.compile(loss='categorical_crossentropy',   # the loss function 
                  optimizer='adam',                  # the SGD algorithm
                  metrics=['accuracy'])              # training returns accuracy and loss values
    
    if name: model.name = name
    return model

<span style="color:Blue; font-family:arial; font-size:11pt;">
Complete the cell below:<br>
- Define <span style="font-family:monospace; background-color: #EBEBEB;">&thinsp;model&thinsp;</span>, the neural network created with a call to <span style="font-family:monospace; background-color: #EBEBEB;">&thinsp;build_NN&thinsp;</span> without giving any seed,<br>
- Display infos on the model with  a call to <span style="font-family:monospace; background-color: #EBEBEB;">&thinsp;model.summary&thinsp;</span>.
</span>

# 4 - Elementary tests of the reproducibility of the model training...

<span style="color:Sienna; font-family:arial; font-size:12pt;">
The training of a neural network is a stochastic process: perfect reproducibility of the training may be difficult to achieve...<br><br>
To highlight the question of reproducibility we run simple loops of trainings with different strategies:<br>
    $\leadsto$ at each lap loop: built the model (no SEED), train once & evaluat the model<br>
    $\leadsto$ built & save (weights) the model (no SEED) once, then at each lap loop: load the model weights, train once & evaluate<br>
    $\leadsto$ built & save the model (no SEED) once, then at each lap loop: load the full model (structure & weights), train once & evaluate<br>
    $\leadsto$  at each lap loop: built the model (SEED fixed), train once & evaluate<br>
    $\leadsto$ built & save the model (SEED fixed) once, then at each lap loop: load the full model, train once and evaluate<br>
</span>

### 4.1 Build the model (no SEED) at each training: not reproducible

We build the model without seed before each training: 

In [ ]:
print(cpu_gpu())
for _ in range(5):
    model = build_DNN(NB_PIXEL, NB_NEURON, NB_CLASS)  # Build a new model without setting seed
    model.fit(x_train, y_train, epochs=1, batch_size=32, validation_data=(x_valid, y_valid), verbose=2)    

The metrics are close together but not exactly the same.<br>
$\leadsto$ This is particularly problematic for `val_accuracy` and `val_loss` used to evaluate the training after each epoch <span style="font-size:16pt;">🙁</span>.

### 4.2 Build and save the model weights (no SEED), then load the model weights at each training: not reproducible

We build the model without seed, save its weights:

In [ ]:
model = build_DNN(NB_PIXEL, NB_NEURON, NB_CLASS)
model.save_weights('model/DNN_noseed.weights.h5')

and then load the model weights before each training: 

In [ ]:
print(cpu_gpu())
for _ in range(5):
    model.load_weights('model/DNN_noseed.weights.h5') # reload the inital model weights
    hist = model.fit(x_train, y_train, epochs=1, batch_size=32, validation_data=(x_valid, y_valid), verbose=2)    

The training is still non-reproducible <span style="font-size:16pt;">😟</span>

### 4.3 Build and save the full model (no SEED) then load the full model at each training: not reproducible

We save the **structure** & **weights** of a model built without setting the seed:

In [ ]:
model = build_DNN(NB_PIXEL, NB_NEURON, NB_CLASS)
model.save('model/DNN_noseed.keras')

Now we load the full model (structure & weights) before each training:

In [ ]:
print(cpu_gpu())
for _ in range(5):
    model = tf.keras.models.load_model('model/DNN_noseed.keras') # reload the inital model weights
    hist = model.fit(x_train, y_train, epochs=1, batch_size=32, validation_data=(x_valid, y_valid), verbose=2)    

The training is still non-reproducible <span style="font-size:16pt;">😟</span>

### 4.4 Build the model (SEED fixed) at each training: reproducible

Now we build the model with the **same fixed seed** at each training: 

In [ ]:
print(cpu_gpu())
for _ in range(5):
    model = build_DNN(NB_PIXEL, NB_NEURON, NB_CLASS, seed=1234)  # Build a new model with seed set
    hist = model.fit(x_train, y_train, epochs=1, batch_size=32, validation_data=(x_valid, y_valid), verbose=2)    

If we set the seed to the same value when building the model the 4 training metrics are perfectly reproducible <span style="font-size:16pt;">😃</span>.

### 4.5 Build and save the full model (SEED fixed), then load the full model at each training: reproducible

To save some computing work, we can try to build the model with a fixed seed only once, save the full model (structure & weights) and just load the full initial model (structure & weights) before each training:

In [ ]:
model = build_DNN(NB_PIXEL, NB_NEURON, NB_CLASS, seed=1234)
model.save('model/DNN_seed-1234.keras')

In [ ]:
print(cpu_gpu())
for _ in range(5):
    model = tf.keras.models.load_model('model/DNN_seed-1234.keras') # reload the model structure & weights 
    hist = model.fit(x_train, y_train, epochs=1, batch_size=32, validation_data=(x_valid, y_valid), verbose=2)    

OK, the successive traings are reproducible <span style="font-size:16pt;">😃</span><br>
What if we build again the model later (for example, we re-run the notebook after closing it)? 

In [ ]:
model = build_DNN(NB_PIXEL, NB_NEURON, NB_CLASS, seed=1234)
model.save('model/DNN_seed-1234.keras')

In [ ]:
print(cpu_gpu())
for _ in range(5):
    model = tf.keras.models.load_model('model/DNN_seed-1234.keras') # reload the model structure & weights 
    hist = model.fit(x_train, y_train, epochs=1, batch_size=32, validation_data=(x_valid, y_valid), verbose=2)    

As far as the seed is set to the same value, the 4 training metrics remain reproducible and keep their values <span style="font-size:16pt;">🤩</span>.

## Conclusion

<span style="color:Sienna; font-family:arial; font-size:12pt;">
To ensure the reproducibility of the training we can:<br>
$\leadsto$ <b>set the seed</b> when building the model (and everywhere else where required)<br>
$\leadsto$ Save the full model (<b>structure & weights</b>)<br>
$\leadsto$ Load the <b>full model</b> (structure and weights) with the <i>tf.keras.models.load_model</i> function befor each training.<br><br>
</spawn>

# 5 - Reproducibility of the full taining of the model...

Now let's see how to apply the previous results (elementary test, only 1 epoch of training) to a complete training over several epochs.

## 5.1 - Build a new model (no SEED) before each training of 15 epochs: not reproducible

<span style="color:Blue; font-family:arial; font-size:11pt;">
Complete the cell below to write a loop that runs the following steps 5 times:<br>
- Define <span style="font-family:monospace; background-color: #EBEBEB;">&thinsp;model&thinsp;</span>, the neural network created by a call to <span style="font-family:monospace; background-color: #EBEBEB;">&thinsp;build_DNN&thinsp;</span> without specifying a value for the seed<br>
- Add to the list  <span style="font-family:monospace; background-color: #EBEBEB;">&thinsp;H&thinsp;</span> the data returned by the call to  <span style="font-family:monospace; background-color: #EBEBEB;">&thinsp;model.fit&thinsp;</span> over 15 epochs, with a batch size of 32 (you can also pass the argument verbose=0 to reduce the displayed infos).
</span>

In [ ]:
H  = []       # to store each traing metrics
t0 = time()   # start chrono...

print(f"Loop ", end="")
for i in range(5):
    print(f" #{i+1}", end="")

    # <your code here>...




    # </your code here>

print(f' Total Train {elapsed_time_since(t0)}')   

To plot accuracy & loss versus epochs we use the `plot_loss_accuracy` function from the local `utils.tools` module.

Let's plot the accuracy & loss curves for training and evaluation with the legends of the 5 plots:

In [ ]:
plot_loss_accuracy(H, message='Model (no SEED) built at each training (15 epochs)', single_legend=False)

Now we plot only validation accuracy & loss versus epochs with the legends of the 5 plots:

In [ ]:
plot_loss_accuracy(H, message='Model (no SEED) built at each training (15 epochs)', plot_train=False, single_legend=False)

The `val_accuracy` and `val_loss` curves differ for each of the trainings.<br>
$\leadsto$ It is a problem when we train the model with the __early stoppping__ extension: the training would stop at a different epoch if we run the training many times.

## 5.2 -Build a new model (fixed SEED) befor each training of 15 epochs: reproducible

<span style="color:Blue; font-family:arial; font-size:11pt;">
Complete the cell below to write a loop that runs the following steps 5 times:<br>
- Define <span style="font-family:monospace; background-color: #EBEBEB;">&thinsp;model&thinsp;</span>, the neural network created by a call to <span style="font-family:monospace; background-color: #EBEBEB;">&thinsp;build_DNN&thinsp;</span> with the argument <span style="font-family:monospace; background-color: #EBEBEB;">&thinsp;seed=1234&thinsp;</span><br>
- Add to the list <span style="font-family:monospace; background-color: #EBEBEB;">&thinsp;H&thinsp;</span> the data returned by the call to <span style="font-family:monospace; background-color: #EBEBEB;">&thinsp;model.fit&thinsp;</span> over 15 epochs, with a batch size of 32 (you can also pass the argument verbose=0 to reduce the displayed infos).
</span>

In [ ]:
H  = []       # to store each traing metrics
t0 = time()   # start chrono...

print(f"Loop ", end="")
for i in range(5):
    print(f" #{i+1}", end="")
    
    # <your code here>...




    # </your code here>

print(f' Total Train {elapsed_time_since(t0)}')

Let's plot the accuracy & loss curves for training and evaluation:

In [ ]:
plot_loss_accuracy(H, message='Model (fixed SEED) built at each training (15 epochs)', single_legend=False)

$\leadsto$ the repoducibility is perfect <span style="font-size:16pt;">😃</span>

## 5.3 - Build and save the model (fixed SEED) once, then load the model for each training (15 epochs): reproducible

To save some computing work, we can build the model with a fixed seed once, save it (structure & weights) and then reload the full initial model (structure & weights) before each training:

<span style="color:Blue; font-family:arial; font-size:11pt;">
Complete the cell below:<br>
- Define <span style="font-family:monospace; background-color: #EBEBEB;">&thinsp;model&thinsp;</span>, the neural network created by a call to <span style="font-family:monospace; background-color: #EBEBEB;">&thinsp;build_DNN&thinsp;</span> includind the argument <span style="font-family:monospace; background-color: #EBEBEB;">&thinsp;seed=1234&thinsp;</span><br>
- Save the model with a call to <span style="font-family:monospace; background-color: #EBEBEB;">&thinsp;model.save&thinsp;</span> giving <span style="font-family:monospace; background-color: #EBEBEB;">&thinsp;'model/DNN_seed-1234.keras'&thinsp;</span> as the name of file
</span>

In [ ]:
model = build_DNN(NB_PIXEL, NB_NEURON, NB_CLASS, seed=1234)
model.save('model/DNN_seed-1234.keras')

<span style="color:Blue; font-family:arial; font-size:11pt;">
Complete the cell below to write a loop that runs the following steps 5 times:<br>
- Define <span style="font-family:monospace; background-color: #EBEBEB;">&thinsp;model&thinsp;</span> by loading the previous model file with a call to <span style="font-family:monospace; background-color: #EBEBEB;">&thinsp;tf.keras.models.load_model&thinsp;</span> <br>
- Add to the list <span style="font-family:monospace; background-color: #EBEBEB;">&thinsp;H&thinsp;</span> the data returned by the call to <span style="font-family:monospace; background-color: #EBEBEB;">&thinsp;model.fit&thinsp;</span> over 15 epochs, with a batch size of 32 (you can also pass the argument verbose=0 to reduce the displayed infos).
</span>

In [ ]:
H  = []       # to store each traing metrics
t0 = time()   # start chrono...

print(f"Loop ", end="")
for i in range(5):
    print(f" #{i+1}", end="")
                
    # <your code here>...




    # </your code here>

print(f' Total Train {elapsed_time_since(t0)}')   

Let's plot the accuracy & loss curves for training and evaluation:

In [ ]:
plot_loss_accuracy(H, message='Model (fixed SEED) reloaded (structure & weights) at each training', single_legend=False)

$\leadsto$ the repoducibility is perfect <span style="font-size:16pt;">😃</span>

# Conclusion

<span style="color:maroon; font-family:arial; font-size:12pt;">

What you have learned in this notebook:
<ul>
  <li>How to load the MNIST databank with the <b>keras</b> module</li>
  <li>How to pre-process and check the input dataset before the training</li>
  <li>Why are there issues to get reproducible trainings and what are the solutions</li>
</ul>

$\leadsto$ in the next notebook <i>02-Trainning_evaluation.ipynb</i>, you will learn how to evaluate the performance<br>
of a trained model and how to apply regularization techniques to avoid overfit.
</span>    